# Causalidade — Perfil Gestacional e Tipo de Gravidez

Dois estudos causais com dados extraídos do texto clínico livre:

## Estudo A — Notação Obstétrica (G/P/A)
**Pergunta:** Grande multípara (G≥5) ou gestante com abortos anteriores (A≥1)
têm maior risco causal de retorno em 30 dias?

## Estudo B — Tipo de Gestação (GUTT/GURT/GGUT/etc.)
**Pergunta:** Gestação de risco classificada como GURT/GGUT tem efeito causal
sobre duração da internação e retorno?

Glossário de abreviações encontradas no texto:
  GU  = Gestação Única          GG  = Gestação Gemelar    GT = Gestação Tripla
  TT  = a Termo                 PT  = Pré-Termo            RT = Risco a Termo
  GUTT = GU + Termo             GURT = GU + Risco a Termo
  GGUT = GG + Única a Termo     GUPT = GU + Pré-Termo  (variante mais comum: 928 registros)
  GGPT = GG + Pré-Termo         GGRT = GG + Risco a Termo

Método: AIPW (duplamente robusto) com bootstrap IC95%

In [1]:
from __future__ import annotations
import re
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm as _norm

warnings.filterwarnings("ignore")

ROOT     = Path("..").resolve()
PARQUET  = ROOT / "data" / "gestante_quadro_clinico.parquet"
LLM_PARQ = ROOT / "out"  / "llm_extracted_20k.parquet"
OUT_DIR  = ROOT / "out" / f"causal_gestacao_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
PLOT_DIR = OUT_DIR / "plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Carga e Extração de Padrões do Texto Clínico

In [2]:
import duckdb as _ddb
_DUCKDB        = ROOT / "data" / "gestante.duckdb"
_PARQUET_ENRIQ = ROOT / "data" / "gestante_quadro_clinico_enriquecido.parquet"

_tabs = _ddb.connect(str(_DUCKDB)).execute("SHOW TABLES").df()["name"].tolist()
if "gestante_enriquecido" in _tabs:
    _con = _ddb.connect(str(_DUCKDB))
    df = _con.execute("SELECT * FROM gestante_enriquecido").df()
    _con.close()
    df.columns = df.columns.str.lower()
    print(f"DuckDB gestante_enriquecido carregado: {len(df):,} registros")
elif _PARQUET_ENRIQ.exists():
    df = pd.read_parquet(_PARQUET_ENRIQ)
    df.columns = df.columns.str.lower()
    print(f"Parquet enriquecido carregado: {len(df):,} registros")
else:
    # fallback: processa do zero
    import sys; sys.path.insert(0, str(ROOT / "src"))
    from build_obstetric_features import enrich_obstetric
    df = pd.read_parquet(PARQUET)  # use whatever PARQUET variable exists in the file
    df.columns = df.columns.str.lower()
    df = df[df["is_recem_nascido"] == 0].copy()
    LLM_PARQUET = ROOT / "out" / "llm_extracted_20k.parquet"
    df = enrich_obstetric(df, llm_parquet=str(_DUCKDB))
    df = df[(df.flag_neonatal == 0)].copy()
    print(f"Processado do zero: {len(df):,} registros")

df["dt_atendimento"] = pd.to_datetime(df["dt_atendimento"], errors="coerce")
df["ano"] = df["dt_atendimento"].dt.year

# Targets
df["Y_ret30"] = (df["sn_retorno_30d"].fillna("N") == "S").astype(int)
df["Y_obito"] = (df["sn_obito"].fillna("N") == "S").astype(int)
df["Y_dur"]   = pd.to_numeric(df["duracao_horas_aprox"], errors="coerce")

# Campos de texto disponíveis
TEXT_COLS = ["ds_queixa_principal", "txt_sinais_sintomas",
             "txt_evolucao_medica", "txt_obs_gestacao",
             "txt_just_internacao"]

def concat_text(row) -> str:
    parts = []
    for c in TEXT_COLS:
        v = row.get(c)
        if v and str(v).strip() and str(v) != "nan":
            parts.append(str(v)[:500])
    return " | ".join(parts)

# Monta texto para análises adicionais de regex
print("Concatenando textos por atendimento...")
df["TEXTO"] = df[TEXT_COLS].apply(concat_text, axis=1)
df["TEM_TEXTO"] = (df["TEXTO"].str.len() > 30).astype(int)
print(f"  Com texto: {df['TEM_TEXTO'].sum():,} / {len(df):,}")

## 2. Estudo A — Extração da Notação Obstétrica (G/P/A/C/N)

In [ ]:
# ── Regex robusto para variações brasileiras de prontuário ─────────────────
# Exemplos encontrados: G4P3(3PC), G2/P1/A0, G3N1, G3A0N1,
#                       G4P2PC1A1, G2P0A1, G1, G2PN1

def extrai_gapn(texto: str) -> dict:
    """
    Extrai G (gestações), P (partos normais), C (cesáreas),
    A (abortos), N (nascidos vivos) do texto clínico.
    Retorna dict com valores int ou None.
    """
    if not texto or len(texto) < 3:
        return {}

    t = texto.upper()
    result = {}

    # G: número de gestações (obrigatório como âncora)
    m = re.search(r'\bG(\d{1,2})\b', t)
    if not m:
        return {}
    result["G"] = int(m.group(1))

    # P: partos normais (Pn, PN, P com número)
    mp = re.search(r'\bP[Nn]?(\d{1,2})\b', t)
    if mp:
        result["P"] = int(mp.group(1))

    # PC ou C: cesáreas  (PC2, C1, 2PC, (3PC))
    mc = re.search(r'(?:PC|(?:\d)PC|C)(\d{1,2})|(\d{1,2})\s*PC', t)
    if mc:
        result["C"] = int(mc.group(1) or mc.group(2))

    # A: abortos
    ma = re.search(r'\bA(\d{1,2})\b', t)
    if ma:
        result["A"] = int(ma.group(1))

    # N: nascidos vivos
    mn = re.search(r'\bN[Vv]?(\d{1,2})\b', t)
    if mn:
        result["N"] = int(mn.group(1))

    return result

# Aplica extração
print("Extraindo notação G/P/A/C/N do texto...")
# Usa G/P/A/C já extraídos pelo módulo compartilhado (LLM > regex > estruturado)
df["G_text"] = df["G_obs"]
df["P_text"] = df["P_obs"]
df["C_text"] = df["C_obs"]
df["A_text"] = df["A_obs"]
df["N_text"] = np.nan  # N não está no módulo base

# Fallback regex local para registros sem cobertura
if df["G_text"].isna().mean() > 0.5:
    extraido = df["TEXTO"].apply(extrai_gapn)
    for campo, col in [("G","G_text"),("P","P_text"),("A","A_text"),("C","C_text")]:
        mask = df[col].isna()
        df.loc[mask, col] = extraido[mask].apply(lambda x: x.get(campo))

n_extraidos = df["G_text"].notna().sum()
print(f"  Notação extraída: {n_extraidos:,} atendimentos ({n_extraidos/len(df)*100:.1f}%)")

# Categorias obstétricas derivadas
df["primigesta"]      = (df["G_text"] == 1).astype(float)
df["grande_multipara"]= (df["G_text"] >= 5).astype(float)
df["tem_aborto"]      = (df["A_text"].fillna(0) >= 1).astype(float)
df["tem_cesarea"]     = (df["C_text"].fillna(0) >= 1).astype(float)
df["multipara_aborto"]= ((df["G_text"] >= 2) & (df["A_text"].fillna(0) >= 1)).astype(float)

# Distribuição
print("\nDistribuição da notação extraída:")
for col, lbl in [("G_text","Gestações"),("P_text","Partos"),
                  ("A_text","Abortos"),("C_text","Cesáreas")]:
    vld = df[col].dropna()
    if len(vld): print(f"  {lbl:15s}: n={len(vld):,}  med={vld.median():.0f}  "
                       f"max={vld.max():.0f}  ≥5: {(vld>=5).mean()*100:.1f}%")

## 3. Estudo B — Extração do Tipo de Gestação (GUTT/GURT/GGUT/etc.)

In [ ]:
# Classificação sistemática: Multiplicidade × Risco × Termo
# GU = única, GG = gemelar, GT = tripla
# TT = a termo (≥37s), PT = pré-termo (<37s), RT = risco a termo

def classifica_tipo_gestacao(texto: str) -> str:
    """
    Classifica o tipo de gestação a partir de abreviações clínicas.
    Retorna: GUTT, GURT, GUPT, GGUT, GGPT, OUTRO, NAO_IDENTIFICADO
    """
    if not texto or len(texto) < 5:
        return "NAO_IDENTIFICADO"
    t = texto.upper()

    # Padrões explícitos primeiro (mais específicos)
    padroes = {
        r'\bGURST\b|\bGU\s*RS\b':  "GURST",  # única risco sem termo
        r'\bGURT\b|\bGU\s*RT\b':   "GURT",   # única risco a termo
        r'\bGUPT\b|\bGU\s*PT\b':   "GUPT",   # única pré-termo
        r'\bGUTT\b|\bGU\s*TT\b':   "GUTT",   # única a termo
        r'\bGGUT\b|\bGG\s*UT\b':   "GGUT",   # gemelar única a termo
        r'\bGGPT\b|\bGG\s*PT\b':   "GGPT",   # gemelar pré-termo
        r'\bGGRT\b|\bGG\s*RT\b':   "GGRT",   # gemelar risco a termo
        r'\bGTUT\b|\bGT\s*\b':     "GTUT",   # tripla
    }
    for pat, label in padroes.items():
        if re.search(pat, t):
            return label

    # Inferência por componentes
    mult = None
    if re.search(r'\bGESTAÇÃO\s+ÚNICA|\bGU\b|\bÚNICA\b', t):
        mult = "GU"
    elif re.search(r'\bGEMELAR|\bGG\b|\bGÊMEOS\b|\bGEMINI\b', t):
        mult = "GG"
    elif re.search(r'\bTRIPLA|\bGT\b', t):
        mult = "GT"

    risco  = bool(re.search(r'\bALTO\s*RISCO|\bRISCO\b|\bGUR\b', t))
    termo  = bool(re.search(r'A\s*TERMO|\bTERMO\b|\bTT\b|\b3[7-9][Ss]\b|\b4[01][Ss]\b', t))
    pretmo = bool(re.search(r'PRÉ.TERMO|PRE.TERMO|\bPT\b|\b2[0-9][Ss]\b|\b3[0-6][Ss]\b', t))

    if mult == "GU":
        if risco and termo:   return "GURT"
        if risco:             return "GUR"
        if pretmo:            return "GUPT"
        if termo:             return "GUTT"
    if mult == "GG":
        if pretmo:            return "GGPT"
        if termo:             return "GGUT"
    if mult:
        return f"{mult}_OUTRO"

    # Pelo menos tem algum indicador
    if risco:  return "RISCO_SEM_MULT"
    if termo:  return "TERMO_SEM_MULT"
    if pretmo: return "PRETERMO_SEM_MULT"
    return "NAO_IDENTIFICADO"

# Usa tipo_gestacao já enriquecido pelo módulo (LLM > regex)
df["TIPO_GESTACAO"] = df["tipo_gestacao"]
print(f"Tipo de gestação — fonte: módulo compartilhado (LLM + regex)")

dist_tipo = df["TIPO_GESTACAO"].value_counts()
print("\nDistribuição do tipo de gestação (texto):")
print(dist_tipo.to_string())
dist_tipo.to_csv(OUT_DIR/"distribuicao_tipo_gestacao.csv")

## 4. Análise Descritiva — Risco por Perfil Gestacional

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Retorno 30d por notação G (primigesta → grande multípara)
df_g = df[df["G_text"].notna() & df["G_text"].between(1,8)].copy()
g_ret = df_g.groupby("G_text")["Y_ret30"].agg(["mean","count"]).reset_index()
g_ret.columns = ["G","taxa","n"]
g_ret["taxa"] *= 100

ax = axes[0]
cores_g = ["#2ca02c" if t < 20 else "#ff7f0e" if t < 30 else "#d62728"
           for t in g_ret["taxa"]]
ax.bar(g_ret["G"].astype(str), g_ret["taxa"], color=cores_g)
ax.axhline(df["Y_ret30"].mean()*100, color="black", ls="--", lw=1,
           label=f"Média geral ({df['Y_ret30'].mean()*100:.1f}%)")
ax.set_xlabel("Número de Gestações (G)"); ax.set_ylabel("Retorno 30d (%)")
ax.set_title("Retorno 30d por Número de Gestações")
for i, row in g_ret.iterrows():
    ax.text(i, row["taxa"]+0.3, f"n={int(row['n'])}", ha="center", fontsize=8)
ax.legend(); ax.grid(axis="y", alpha=0.3)

# Retorno 30d por tipo de gestação (top 8, garante GUPT e GGUT)
_top_base = dist_tipo[dist_tipo.index != "NAO_IDENTIFICADO"].head(8).index.tolist()
for _t in ["GUPT", "GGUT"]:
    if _t not in _top_base and _t in dist_tipo.index:
        _top_base.append(_t)
top_tipos = _top_base
df_tipo = df[df["TIPO_GESTACAO"].isin(top_tipos)].copy()
tipo_ret = df_tipo.groupby("TIPO_GESTACAO")["Y_ret30"].agg(["mean","count"]).reset_index()
tipo_ret.columns = ["tipo","taxa","n"]
tipo_ret["taxa"] *= 100
tipo_ret = tipo_ret.sort_values("taxa", ascending=False)

ax2 = axes[1]
cores_t = ["#d62728" if t > 25 else "#ff7f0e" if t > 15 else "#2ca02c"
           for t in tipo_ret["taxa"]]
ax2.barh(tipo_ret["tipo"][::-1], tipo_ret["taxa"][::-1], color=cores_t[::-1])
ax2.axvline(df["Y_ret30"].mean()*100, color="black", ls="--", lw=1)
ax2.set_xlabel("Retorno 30d (%)"); ax2.set_title("Retorno 30d por Tipo de Gestação")
ax2.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR/"01_retorno_por_gestacao.png", dpi=150); plt.show(); plt.close()

tipo_ret.to_csv(OUT_DIR/"retorno_por_tipo_gestacao.csv", index=False)

## 5. Funções de Inferência Causal (AIPW)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from scipy.stats import norm as _norm

def make_preprocessor(X):
    num = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat = [c for c in X.columns if c not in num]
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc",  StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore",
                                                sparse_output=False))]), cat),
    ], remainder="drop")

CONFUNDIDORES_BASE = [
    "idade_anos", "tp_atendimento", "segmento_cid",
    "vl_escore_triagem", "qtd_pre_med", "qtd_evolucoes_documentadas",
    "qtd_pedidos_lab", "duracao_horas_aprox",
]

def aipw_binario(df_ana, T_col, Y_col, confundidores, n_boot=500):
    """AIPW para outcome binário. Retorna ATE em pp + IC95% + p-valor."""
    df_a = df_ana[[T_col, Y_col] + confundidores].copy()
    df_a = df_a.dropna(subset=[T_col, Y_col])
    df_a = df_a[df_a[Y_col].isin([0,1])]

    T = df_a[T_col].values.astype(int)
    Y = df_a[Y_col].values.astype(int)
    CONF = [c for c in confundidores if c in df_a.columns]

    pre = make_preprocessor(df_a[CONF])
    X   = pre.fit_transform(df_a[CONF])

    # Propensity score
    ps_m = GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                       learning_rate=0.05, random_state=42)
    ps_m.fit(X, T)
    ps = ps_m.predict_proba(X)[:,1]
    ps = np.clip(ps, 0.05, 0.95)

    # Outcome model
    X_out = np.hstack([X, T.reshape(-1,1)])
    out_m = GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                        learning_rate=0.05, random_state=42)
    out_m.fit(X_out, Y)

    X1 = np.hstack([X, np.ones((len(X),1))])
    X0 = np.hstack([X, np.zeros((len(X),1))])
    mu1 = out_m.predict_proba(X1)[:,1]
    mu0 = out_m.predict_proba(X0)[:,1]

    sc1 = mu1 + T*(Y-mu1)/(ps+1e-9)
    sc0 = mu0 + (1-T)*(Y-mu0)/(1-ps+1e-9)
    ate = (sc1-sc0).mean()

    rng = np.random.default_rng(42)
    boots = [(sc1[rng.integers(0,len(Y),len(Y))]-sc0[rng.integers(0,len(Y),len(Y))]).mean()
             for _ in range(n_boot)]
    se    = np.std(boots)
    lo,hi = np.percentile(boots,[2.5,97.5])
    pval  = 2*(1-_norm.cdf(abs(ate/se))) if se > 0 else 1.0

    return {
        "n": len(df_a), "n_exp": int(T.sum()),
        "prev_Y": round(Y.mean()*100, 2),
        "ate_pp": round(ate*100, 4),
        "ic_lo_pp": round(lo*100, 4),
        "ic_hi_pp": round(hi*100, 4),
        "se": round(se, 4),
        "p_valor": round(pval, 4),
    }

def aipw_continuo(df_ana, T_col, Y_col, confundidores, n_boot=300):
    """AIPW para outcome contínuo (horas de internação)."""
    df_a = df_ana[[T_col, Y_col] + confundidores].copy()
    df_a = df_a.dropna(subset=[T_col, Y_col])
    df_a = df_a[df_a[Y_col].between(1, 720)]

    T = df_a[T_col].values.astype(int)
    Y = df_a[Y_col].values.astype(float)
    CONF = [c for c in confundidores if c in df_a.columns]

    pre = make_preprocessor(df_a[CONF])
    X   = pre.fit_transform(df_a[CONF])

    ps_m = GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                       learning_rate=0.05, random_state=42)
    ps_m.fit(X, T)
    ps = np.clip(ps_m.predict_proba(X)[:,1], 0.05, 0.95)

    X_out = np.hstack([X, T.reshape(-1,1)])
    out_m = GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                       learning_rate=0.05, random_state=42)
    out_m.fit(X_out, Y)

    X1 = np.hstack([X, np.ones((len(X),1))])
    X0 = np.hstack([X, np.zeros((len(X),1))])
    mu1 = out_m.predict(X1)
    mu0 = out_m.predict(X0)

    sc1 = mu1 + T*(Y-mu1)/(ps+1e-9)
    sc0 = mu0 + (1-T)*(Y-mu0)/(1-ps+1e-9)
    ate = (sc1-sc0).mean()

    rng = np.random.default_rng(42)
    boots = [(sc1[rng.integers(0,len(Y),len(Y))]-sc0[rng.integers(0,len(Y),len(Y))]).mean()
             for _ in range(n_boot)]
    se    = np.std(boots)
    lo,hi = np.percentile(boots,[2.5,97.5])
    pval  = 2*(1-_norm.cdf(abs(ate/se))) if se > 0 else 1.0

    return {
        "n": len(df_a), "n_exp": int(T.sum()),
        "Y_media_exp":  round(Y[T==1].mean(),1),
        "Y_media_ctrl": round(Y[T==0].mean(),1),
        "ate_h":   round(ate,2),
        "ic_lo_h": round(lo,2),
        "ic_hi_h": round(hi,2),
        "se":      round(se,4),
        "p_valor": round(pval,4),
    }

## 6. Estudo A — Grande Multípara (G≥5) → Retorno 30d

In [ ]:
print("=" * 60)
print("ESTUDO A1 — Grande Multípara (G≥5) → Retorno 30d")
print("=" * 60)

df_gm = df[df["G_text"].notna()].copy()
df_gm["T_grande_multipara"] = (df_gm["G_text"] >= 5).astype(int)

conf_gm = [c for c in CONFUNDIDORES_BASE if c in df_gm.columns]

r_a1 = aipw_binario(df_gm, "T_grande_multipara", "Y_ret30", conf_gm)
print(f"\n  N análise:       {r_a1['n']:,}")
print(f"  Grandes mult.:   {r_a1['n_exp']:,} ({r_a1['n_exp']/r_a1['n']*100:.1f}%)")
print(f"  Prevalência Y:   {r_a1['prev_Y']}%")
print(f"  ATE (AIPW):     {r_a1['ate_pp']:+.2f} pp")
print(f"  IC95%:          [{r_a1['ic_lo_pp']:+.2f}, {r_a1['ic_hi_pp']:+.2f}]")
print(f"  p-valor:        {r_a1['p_valor']:.4f}")

## 7. Estudo A — Aborto Anterior (A≥1) → Retorno 30d

In [ ]:
print("\n" + "=" * 60)
print("ESTUDO A2 — Aborto Anterior (A≥1) → Retorno 30d")
print("=" * 60)

df_ab = df[df["A_text"].notna()].copy()
df_ab["T_aborto_ant"] = (df_ab["A_text"] >= 1).astype(int)

r_a2 = aipw_binario(df_ab, "T_aborto_ant", "Y_ret30",
                     [c for c in CONFUNDIDORES_BASE if c in df_ab.columns])
print(f"\n  N análise:       {r_a2['n']:,}")
print(f"  Com aborto ant.: {r_a2['n_exp']:,} ({r_a2['n_exp']/r_a2['n']*100:.1f}%)")
print(f"  ATE (AIPW):     {r_a2['ate_pp']:+.2f} pp  IC95% [{r_a2['ic_lo_pp']:+.2f}, {r_a2['ic_hi_pp']:+.2f}]  p={r_a2['p_valor']:.4f}")

## 8. Estudo A — Grande Multípara → Duração da Internação

In [ ]:
print("\n" + "=" * 60)
print("ESTUDO A3 — Grande Multípara → Duração Internação (horas)")
print("=" * 60)

df_dur_gm = df[df["G_text"].notna() & df["Y_dur"].notna()].copy()
df_dur_gm["T_grande_multipara"] = (df_dur_gm["G_text"] >= 5).astype(int)

r_a3 = aipw_continuo(df_dur_gm, "T_grande_multipara", "Y_dur",
                      [c for c in CONFUNDIDORES_BASE if c in df_dur_gm.columns])
print(f"\n  Duração média — grandes mult.: {r_a3['Y_media_exp']:.1f}h")
print(f"  Duração média — demais:        {r_a3['Y_media_ctrl']:.1f}h")
print(f"  ATE (AIPW):     {r_a3['ate_h']:+.1f}h  IC95% [{r_a3['ic_lo_h']:+.1f}, {r_a3['ic_hi_h']:+.1f}]  p={r_a3['p_valor']:.4f}")

## 9. Estudo B — GURT (Gestação Única de Risco a Termo) → Retorno 30d

In [ ]:
print("\n" + "=" * 60)
print("ESTUDO B1 — GURT vs GUTT → Retorno 30d")
print("(Gestação de Risco vs Gestação Normal a Termo)")
print("=" * 60)

df_tipo = df[df["TIPO_GESTACAO"].isin(["GURT","GUTT","GUR","GUPT"])].copy()
df_tipo["T_risco"] = (df_tipo["TIPO_GESTACAO"].isin(["GURT","GUR"])).astype(int)

if len(df_tipo) > 100:
    r_b1 = aipw_binario(df_tipo, "T_risco", "Y_ret30",
                         [c for c in CONFUNDIDORES_BASE if c in df_tipo.columns])
    print(f"\n  N análise:  {r_b1['n']:,}")
    print(f"  GURT/GUR:   {r_b1['n_exp']:,}  vs  GUTT: {r_b1['n']-r_b1['n_exp']:,}")
    print(f"  ATE (AIPW): {r_b1['ate_pp']:+.2f} pp  IC95% [{r_b1['ic_lo_pp']:+.2f}, {r_b1['ic_hi_pp']:+.2f}]  p={r_b1['p_valor']:.4f}")
else:
    r_b1 = None
    print("  Amostra insuficiente (<100). Verifique cobertura do texto.")

## 10. Estudo B — Gemelar (GG) → Retorno 30d

In [ ]:
print("\n" + "=" * 60)
print("ESTUDO B2 — Gestação Gemelar (GG) vs Única (GU) → Retorno 30d")
print("=" * 60)

df_mult = df[df["TIPO_GESTACAO"].str.startswith(("GG","GU"), na=False)].copy()
df_mult["T_gemelar"] = df_mult["TIPO_GESTACAO"].str.startswith("GG").astype(int)

if len(df_mult) > 100 and df_mult["T_gemelar"].sum() > 20:
    r_b2 = aipw_binario(df_mult, "T_gemelar", "Y_ret30",
                         [c for c in CONFUNDIDORES_BASE if c in df_mult.columns])
    print(f"\n  Gemilares: {r_b2['n_exp']:,}  vs  Únicas: {r_b2['n']-r_b2['n_exp']:,}")
    print(f"  ATE (AIPW): {r_b2['ate_pp']:+.2f} pp  IC95% [{r_b2['ic_lo_pp']:+.2f}, {r_b2['ic_hi_pp']:+.2f}]  p={r_b2['p_valor']:.4f}")
else:
    r_b2 = None
    print("  Amostra insuficiente.")

## 11. Estudo B — Pré-Termo (GUPT/GGPT) → Duração da Internação

In [ ]:
print("\n" + "=" * 60)
print("ESTUDO B3 — Gestação Pré-Termo → Duração Internação (horas)")
print("=" * 60)

df_pt = df[df["TIPO_GESTACAO"].isin(["GUPT","GGPT","PRETERMO_SEM_MULT","GUTT","GGUT"])].copy()
df_pt["T_pretermo"] = df_pt["TIPO_GESTACAO"].isin(["GUPT","GGPT","PRETERMO_SEM_MULT"]).astype(int)
df_pt = df_pt[df_pt["Y_dur"].notna()]

if len(df_pt) > 100 and df_pt["T_pretermo"].sum() > 20:
    r_b3 = aipw_continuo(df_pt, "T_pretermo", "Y_dur",
                          [c for c in CONFUNDIDORES_BASE if c in df_pt.columns])
    print(f"\n  Duração — pré-termo: {r_b3['Y_media_exp']:.1f}h  vs  a termo: {r_b3['Y_media_ctrl']:.1f}h")
    print(f"  ATE (AIPW): {r_b3['ate_h']:+.1f}h  IC95% [{r_b3['ic_lo_h']:+.1f}, {r_b3['ic_hi_h']:+.1f}]  p={r_b3['p_valor']:.4f}")
else:
    r_b3 = None
    print("  Amostra insuficiente.")

## 12. Visualização Consolidada — Forest Plot Causal

In [ ]:
estudos = [
    ("A1 — Grande Multípara → Retorno 30d",    r_a1, "pp"),
    ("A2 — Aborto Anterior → Retorno 30d",      r_a2, "pp"),
    ("A3 — Grande Multípara → Duração (h)",     r_a3, "h"),
]
if r_b1: estudos.append(("B1 — GURT vs GUTT → Retorno 30d", r_b1, "pp"))
if r_b2: estudos.append(("B2 — Gemelar → Retorno 30d",       r_b2, "pp"))
if r_b3: estudos.append(("B3 — Pré-Termo → Duração (h)",     r_b3, "h"))

fig, ax = plt.subplots(figsize=(10, max(4, len(estudos)*1.2)))
y_pos = range(len(estudos))

for i, (lbl, r, unit) in enumerate(estudos):
    if r is None: continue
    ate = r.get("ate_pp") or r.get("ate_h")
    lo  = r.get("ic_lo_pp") or r.get("ic_lo_h")
    hi  = r.get("ic_hi_pp") or r.get("ic_hi_h")
    cor = "#d62728" if ate > 0 else "#2ca02c"
    ax.errorbar(ate, i, xerr=[[ate-lo],[hi-ate]],
                fmt="o", color=cor, capsize=5, markersize=7)
    p = r["p_valor"]
    sig = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns"
    ax.text(hi+0.3, i, f"p={p:.3f} {sig}", va="center", fontsize=8)

ax.axvline(0, color="black", lw=0.8, ls="--")
ax.set_yticks(list(y_pos))
ax.set_yticklabels([e[0] for e in estudos], fontsize=9)
ax.set_xlabel("ATE (pp = pontos percentuais | h = horas)")
ax.set_title("Forest Plot — Estudos Causais Gestacionais (AIPW)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR/"02_forest_causal_gestacao.png", dpi=150)
plt.show(); plt.close()

## 13. Export e Resumo

In [ ]:
import json

resumo = {
    "data": datetime.now().isoformat(),
    "ESTUDO_A1_grande_multipara_retorno30d": r_a1,
    "ESTUDO_A2_aborto_anterior_retorno30d":  r_a2,
    "ESTUDO_A3_grande_multipara_duracao_h":  r_a3,
    "ESTUDO_B1_GURT_vs_GUTT_retorno30d":    r_b1,
    "ESTUDO_B2_gemelar_retorno30d":          r_b2,
    "ESTUDO_B3_pretermo_duracao_h":          r_b3,
}

(OUT_DIR/"resumo_causal_gestacao.json").write_text(
    json.dumps(resumo, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8"
)

df[["cd_atendimento","G_text","P_text","A_text","C_text","N_text",
    "TIPO_GESTACAO","Y_ret30","Y_dur"]].to_parquet(
    OUT_DIR/"features_gestacionais.parquet", index=False
)

print(f"\n✅ Concluído — {OUT_DIR}")
print(f"   resumo_causal_gestacao.json")
print(f"   features_gestacionais.parquet")
print(f"   plots/ (forest plot + distribuição)")

print(f"\n{'='*60}")
print("RESUMO EXECUTIVO")
print(f"{'='*60}")
for nome, r, unit in estudos:
    if r is None: continue
    ate = r.get("ate_pp") or r.get("ate_h")
    lo  = r.get("ic_lo_pp") or r.get("ic_lo_h")
    hi  = r.get("ic_hi_pp") or r.get("ic_hi_h")
    p   = r["p_valor"]
    sig = "SIGNIFICATIVO" if p < 0.05 else "não significativo"
    print(f"\n  {nome}")
    print(f"  ATE = {ate:+.2f} {unit}  IC95%[{lo:+.2f},{hi:+.2f}]  p={p:.4f}  → {sig}")